# Embodied Motion Flow - Kaggle Full Run
Training lungo AIST++ con CFG/EMA e showcase Stardust 15s.


In [ ]:
import os, sys, subprocess, glob, json, shutil
from pathlib import Path
REPO_URL = "https://github.com/Blackhand01/Humanoid-Motion-Diffusion"
WORKDIR = Path("/kaggle/working/Embodied-Motion-Diffusion")
PYTHON = sys.executable
def sh(cmd: list[str]) -> None:
    print("$", " ".join(cmd)); subprocess.run(cmd, check=True)
print("Python:", PYTHON)
print("Workdir:", WORKDIR)
print("Repo:", REPO_URL)
print("Mode: full fresh training")
print("-")


In [ ]:
if WORKDIR.exists():
    shutil.rmtree(WORKDIR)
sh(["git", "clone", REPO_URL, str(WORKDIR)])
os.chdir(WORKDIR)
packages = "numpy PyYAML scikit-learn tqdm matplotlib pandas imageio imageio-ffmpeg librosa soundfile".split()
sh([PYTHON, "-m", "pip", "install", "-q", *packages])
print("Repo ready:", Path.cwd())
print("Runtime deps installed")
print("Torch version check follows")
sh([PYTHON, "-c", "import torch; print(torch.__version__, torch.cuda.is_available())"])
print("Step 2 done")
print("-")


In [ ]:
DATASET_BASE = Path("/kaggle/input/datasets/stefanoroybisignano/aist-smpl-clean/aist-smpl-clean")
STARDUST = Path("/kaggle/input/datasets/stefanoroybisignano/stardustdata/stardust.mp3")
if not DATASET_BASE.exists():
    found = glob.glob("/kaggle/input/**/motions", recursive=True)
    if found: DATASET_BASE = Path(found[0]).parent
paths = {"AISTPP_ROOT": DATASET_BASE/"motions", "AISTPP_SPLIT_ROOT": DATASET_BASE/"splits"}
paths["AISTPP_IGNORE_LIST"] = DATASET_BASE/"ignore_list.txt"
paths["AISTPP_AUDIO_ROOT"] = DATASET_BASE/"audio"; paths["SHOWCASE_TRACK"] = STARDUST
for key, value in paths.items(): os.environ[key] = str(value)
print(json.dumps({k: f"{v} :: {Path(v).exists()}" for k, v in os.environ.items() if k in paths}, indent=2))
print("Data paths mapped")
print("-")


In [ ]:
import yaml; cfg = yaml.safe_load(Path("config.yaml").read_text())
cfg["project"]["output_dir"] = "/kaggle/working/outputs"
cfg["device"]["preference"] = "auto"
cfg["data"]["sequence_length"] = 120
cfg["data"]["batch_size"] = 32
cfg["data"]["aist"]["root_dir"] = os.environ["AISTPP_ROOT"]
cfg["data"]["aist"]["split_dir"] = os.environ["AISTPP_SPLIT_ROOT"]
cfg["data"]["aist"]["ignore_list_path"] = os.environ["AISTPP_IGNORE_LIST"]
cfg["data"]["aist"]["max_files_per_split"] = None
cfg["data"]["aist"]["clip_stride"] = 15
cfg["audio"]["root_dir"] = os.environ["AISTPP_AUDIO_ROOT"]
print("Patched data config")


In [ ]:
cfg["audio"]["cache_dir"] = "/kaggle/working/outputs/audio_features"
cfg["training"]["epochs"] = 180
cfg["training"]["accumulation_steps"] = 2
cfg["training"]["warmup_epochs"] = 5
cfg["training"]["save_every_epochs"] = 10
cfg["training"]["auto_resume"] = False
cfg["training"]["mixed_precision"] = True
cfg["training"]["log_every_steps"] = 100
cfg["inference"]["guidance_scale"] = 1.35
cfg["inference"]["sliding_window_frames"] = 120
cfg["inference"]["prefix_frames"] = 60
print("Patched training/inference config")


In [ ]:
cmd = [PYTHON, "kaggle_showcase_main.py", "--config", "config.kaggle.full.yaml", "--fresh-start", "--zip-path", "/kaggle/working/embodied_motion_flow_showcase.zip"]
cfg["showcase"]["track_path"] = os.environ["SHOWCASE_TRACK"]
cfg["showcase"]["clip_start_seconds"] = 46.0
cfg["showcase"]["clip_duration_seconds"] = 15.0
cfg["showcase"]["output_dir"] = "/kaggle/working/outputs/showcase"
cfg["showcase"]["viral_fps"] = 30
cfg["showcase"]["research_fps"] = 30
cfg["showcase"]["render_dpi"] = 140
Path("config.kaggle.full.yaml").write_text(yaml.safe_dump(cfg, sort_keys=False))
print("Wrote config.kaggle.full.yaml")
print("Expected: thousands of clips, not 273")
print("-")


In [ ]:
cmd = [PYTHON, "kaggle_showcase_main.py", "--config", "config.kaggle.full.yaml", "--fresh-start", "--zip-path", "/kaggle/working/embodied_motion_flow_showcase.zip"]
print("Command:", " ".join(cmd))
print("This should run much longer than the previous 5 minute run")
print("Key targets: TSI < 1.0, JLVR < 0.10")
print("Outputs will be under /kaggle/working/outputs")
print("Starting training and showcase")
sh(cmd)
print("Pipeline completed")
print("-")
print("-")
print("-")
print("-")


In [ ]:
from IPython.display import FileLink, Video, display
zip_path = Path("/kaggle/working/embodied_motion_flow_showcase.zip")
out = Path("/kaggle/working/outputs/showcase")
metrics = Path("/kaggle/working/outputs/metrics/evaluation_metrics.json")
print(metrics.read_text() if metrics.exists() else "missing metrics")
print("ZIP:", zip_path, "exists=", zip_path.exists())
if zip_path.exists(): display(FileLink(str(zip_path)))
for p in sorted(out.glob("*")): print(p)
viral = out / "stardust_0046_0101_viral.mp4"
research = out / "stardust_0046_0101_research.mp4"
if viral.exists(): display(Video(str(viral), embed=True, width=480))
if research.exists(): display(Video(str(research), embed=True, width=720))
